In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
data_path = "/content/drive/MyDrive/DeepfakeData"

if os.path.exists(data_path):
    print("✅ Successfully accessed data from the previous account!")
    print(f"Items found: {os.listdir(data_path)}")
else:
    print("❌ Cannot find the folder. Ensure the shortcut was added to 'My Drive'.")

In [ ]:
!pip install dlib

In [ ]:
import os
import cv2
import dlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, roc_auc_score,
                             recall_score, precision_score)

from tensorflow.keras.utils import to_categorical, Sequence
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception, MobileNet, ResNet50
from tensorflow.keras.layers import (Dense, Flatten, TimeDistributed, LSTM,
                                     Dropout, GlobalAveragePooling2D, Input)
from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

# --- 1. CONFIGURATION PARAMETERS ---


# Image and Video Processing Settings
OUTPUT_FRAME_SIZE = (224, 224) # Dimensions for the input layer
FRAME_COUNT = 15               # Number of frames to extract per video
MAX_VIDEOS = 200               # Limit total videos to manage memory/time
BATCH_SIZE = 4                 # Training batch size
PADDING_FACTOR = 1.3           # Multiplier for cropping the face area

# Hyperparameters
LR_FINE_TUNE = 1e-6            # Initial Learning Rate for the fine-tuning phase

In [ ]:
# Initialize dlib's face detector
face_detector_dlib = dlib.get_frontal_face_detector()

def extract_frames(video_path, output_size=OUTPUT_FRAME_SIZE, frame_count=FRAME_COUNT, padding_factor=1.3):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        return np.array([])

    step = max(total_frames // frame_count, 1)

    for i in range(frame_count):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_detector_dlib(gray)

        frame_h, frame_w, _ = frame.shape # Original frame dimensions

        if len(faces) > 0:
            d = faces[0]

            # Calculate face center and dimensions
            y_center, x_center = (d.top() + d.bottom()) // 2, (d.left() + d.right()) // 2
            h, w = d.bottom() - d.top(), d.right() - d.left()

            # Calculate conservative cropping
            h_pad, w_pad = int(h * padding_factor), int(w * padding_factor)

            # Determine new crop coordinates (avoid out-of-bounds errors)
            y1 = max(0, y_center - h_pad // 2)
            y2 = min(frame_h, y_center + h_pad // 2)
            x1 = max(0, x_center - w_pad // 2)
            x2 = min(frame_w, x_center + w_pad // 2)

            cropped_face = frame[y1:y2, x1:x2]

            if cropped_face.size != 0:
                resized_frame = cv2.resize(cropped_face, output_size)
                frames.append(resized_frame)
                continue # Successfully found and cropped

        # If Dlib finds nothing or cropping fails, resize the full frame
        resized_frame = cv2.resize(frame, output_size)
        frames.append(resized_frame)

    cap.release()
    return np.array(frames) if len(frames) == frame_count else np.array([])

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import gc

# 1. Load data using memory mapping (mmap_mode)
# This keeps the data on the disk and only loads it into RAM when accessed
print("Loading processed data from Drive (Memory-Mapped)...")
data = np.load('/content/drive/MyDrive/DeepfakeData/processed_data.npy', mmap_mode='r')
labels = np.load('/content/drive/MyDrive/DeepfakeData/processed_labels.npy')

print(f"Loaded {data.shape[0]} videos via mmap. RAM usage is still low.")

# 2. Split the dataset (indices only first to save RAM)
indices = np.arange(data.shape[0])
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, labels, test_size=0.3, random_state=42, stratify=labels
)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 3. Load the specific slices into memory
X_train = np.array(data[idx_train], dtype='float32') / 255.0
gc.collect()

X_val = np.array(data[idx_val], dtype='float32') / 255.0
gc.collect()

X_test = np.array(data[idx_test], dtype='float32') / 255.0
gc.collect()

# 4. Label Encoding
y_train_cat = to_categorical(y_train, num_classes=2)
y_val_cat = to_categorical(y_val, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)

print(f"Data ready! Training shape: {X_train.shape}")

# Final cleanup of the mmap pointer
del data
del labels
gc.collect()

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

# --- SPECIFIC VIDEO PATH CONFIGURATION ---
REAL_VIDEO_PATH = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/original/012.mp4"
FAKE_VIDEO_PATH = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/Deepfakes/012_026.mp4"

def illustrate_specific_comparison(real_path, fake_path):
    print(f"🔄 Processing video comparison...")
    print(f"   - Real: {real_path}")
    print(f"   - Fake: {fake_path}")

    # Check for file existence
    if not os.path.exists(real_path):
        print(f"❌ Error: REAL video not found at: {real_path}")
        return
    if not os.path.exists(fake_path):
        print(f"❌ Error: FAKE video not found at: {fake_path}")
        return

    # Helper function to extract a frame
    def get_frame(video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: return None

        # Get frame at 50% duration to ensure face presence
        target_frame = total_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
        ret, frame = cap.read()
        cap.release()

        if ret:
            # Convert from BGR to RGB for correct color display
            return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return None

    # Get frames
    img_real = get_frame(real_path)
    img_fake = get_frame(fake_path)

    if img_real is None or img_fake is None:
        print("❌ Error: Could not read frames from video (file may be corrupted or inaccessible).")
        return

    # Plot comparison
    plt.figure(figsize=(16, 8))

    # Real Image
    plt.subplot(1, 2, 1)
    plt.imshow(img_real)
    plt.title(f"ORIGINAL VIDEO (REAL)\n{os.path.basename(real_path)}", color='green', fontsize=14, fontweight='bold')
    plt.axis('off')

    # Fake Image
    plt.subplot(1, 2, 2)
    plt.imshow(img_fake)
    plt.title(f"FORGED VIDEO (DEEPFAKE)\n{os.path.basename(fake_path)}", color='red', fontsize=14, fontweight='bold')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
    print("✅ Illustration generated successfully.")

# --- EXECUTION ---
illustrate_specific_comparison(REAL_VIDEO_PATH, FAKE_VIDEO_PATH)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import dlib

# Load the face detector model from dlib
detector = dlib.get_frontal_face_detector()

# Read a single frame from a video
# Ensure this path is correct based on your dataset location
video_path = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/original/018.mp4"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if ret:
    # Convert the frame to grayscale for dlib detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    print(f"Detected {len(faces)} face(s) in the frame.")

    # Create a copy of the original frame for visualization
    frame_vis = frame.copy()

    for face in faces:
        # Get coordinates from the dlib rectangle object
        x, y = face.left(), face.top()
        w, h = face.width(), face.height()

        # Draw a bounding box around the detected face
        # We use a thickness of 10 to make it clearly visible on high-res frames
        cv2.rectangle(frame_vis, (x, y), (x + w, y + h), (0, 255, 0), 10)

    # Display the full frame with the bounding box
    plt.figure(figsize=(12, 8))
    # Convert from BGR (OpenCV) to RGB (Matplotlib)
    plt.imshow(cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB))
    plt.title("Face Extraction Process (Full Frame with Bounding Box)")
    plt.axis('off')
    plt.show()
else:
    print("❌ Error: Could not read the video file. Please check the path.")

In [ ]:
# 3. DATA GENERATOR

# Define the augmentation parameters
datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15
)

class VideoDataGenerator(Sequence):
    """
    Custom Data Generator to handle video sequences.
    Applies real-time data augmentation to individual frames.
    """
    def __init__(self, x_set, y_set, batch_size, augmentations=None):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.augment = augmentations

    def __len__(self):
        # Returns the number of batches per epoch
        return int(np.ceil(len(self.x) / self.batch_size))

    def __getitem__(self, idx):
        # Generates one batch of data
        indices = np.arange(idx * self.batch_size, min((idx + 1) * self.batch_size, len(self.x)))
        batch_x, batch_y = self.x[indices], self.y[indices]

        if self.augment:
            # Apply augmentation to every frame in each video of the batch
            return np.array([self.augment_video(video) for video in batch_x]), batch_y

        return batch_x, batch_y

    def augment_video(self, video_frames):
        # Applies the same random transformation logic to each frame in a single video
        return np.array([self.augment.random_transform(frame) for frame in video_frames])

# Initialize the generators for training and validation
train_generator = VideoDataGenerator(X_train, y_train_cat, BATCH_SIZE, datagen)
validation_generator = VideoDataGenerator(X_val, y_val_cat, BATCH_SIZE)

In [ ]:
#2
from tensorflow.keras.layers import Bidirectional, LSTM

def build_finetune_model(input_shape=(FRAME_COUNT, 224, 224, 3)):
    # Base spatial feature extractor
    base_model = Xception(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False

    model = Sequential([
        # 1. Spatial Extraction (Per-frame)
        TimeDistributed(base_model, input_shape=input_shape),

        # 2. Dimensionality Reduction (Focuses on key facial features)
        TimeDistributed(GlobalAveragePooling2D()),

        # 3. Temporal Analysis (Bidirectional for higher confidence)
        Bidirectional(LSTM(128, return_sequences=False)),

        # 4. Classification Head
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(2, activation='softmax')
    ])
    return model

model = build_finetune_model()

In [ ]:
### 5. MODEL TRAINING (Class Weights & Two-Phase Fine-Tuning) ###

# NEW: Calculate Class Weights to handle potential data imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convert to dictionary: {0: weight_0, 1: weight_1}
class_weight_dict = dict(enumerate(class_weights))
print(f"Class Weights to balance data: {class_weight_dict}")

# Optimization Callbacks
callbacks = [
    ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        verbose=1,
        restore_best_weights=True
    )
]

import tensorflow.keras.backend as K
import gc

# Clear memory
K.clear_session()
gc.collect()


# --- PHASE 1: Training Top Layers (Head Only) ---
print("--- PHASE 1: Training Top Layers with Class Weights ---")

# Use a standard learning rate for the new layers
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_top = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weight_dict  # NEW: Apply class weight here
)

# --- PHASE 2: Safer Fine-tuning ---
print("\n--- PHASE 2: Fine-tuning ---")

# Unfreeze the base model (Xception) inside the TimeDistributed wrapper
model.layers[0].layer.trainable = True

# NEW: Keep BatchNormalization layers frozen to maintain statistical stability
for layer in model.layers[0].layer.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Only unfreeze the last 20 layers of Xception to prevent aggressive weight distortion
for layer in model.layers[0].layer.layers[:-20]:
    layer.trainable = False

# Re-compile with an extremely small learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training from the last epoch of Phase 1
history_finetune = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=60,
    initial_epoch=history_top.epoch[-1],
    callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
import pandas as pd
import os

version = "v2_bidirectional_lstm_400vid"
save_path = f"/content/drive/MyDrive/DeepfakeData/v2_bidirectional_lstm_400vid"

if not os.path.exists(save_path):
    os.makedirs(save_path)

# 1. Save the Final Model

model.save(f"{save_path}/deepfake_model_final.keras")

# 2. Save the Weights separately
model.save_weights(f"{save_path}/deepfake_weights.weights.h5")

# 3.Save the Training History to CSV
# This allows to plot the graphs
history_df = pd.concat([pd.DataFrame(history_top.history),
                        pd.DataFrame(history_finetune.history)],
                        ignore_index=True)

history_df.to_csv(f"{save_path}/training_history.csv", index=False)

print(f"✅ Version {version} saved successfully!")
print(f"📍 Location: {save_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Input
from tensorflow.keras.applications import ResNet50, MobileNet, Xception
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

# --- CONFIGURATION ---
THRESHOLD_OPTIMAL = 0.42
EPOCHS_BASELINE = 10
BATCH_SIZE = 8

print("🚀 STARTING MODEL COMPARISON PROCESS (Section 4.4)...\n")

# ==============================================================================
# STEP 1: COLLECT RESULTS FROM PROPOSED MODEL (XCEPTION-LSTM)
# ==============================================================================
print("--- 1. Collecting Results for Proposed Model (Xception-LSTM) ---")

# Data safety check
if 'X_test' not in locals() or 'y_test' not in locals():
    raise ValueError("❌ ERROR: X_test/y_test not found. Please run the data processing cells first!")

# Load the best model or use the current one in memory
try:
    best_model = load_model('/content/drive/MyDrive/DeepfakeData/v2_bidirectional_lstm_400vid/deepfake_model_final.keras')
    print("   -> Successfully loaded model from 'deepfake_model_final.keras'.")
except:
    print("   -> Checkpoint not found. Using the current 'model' variable.")
    best_model = model

# Re-predicting to ensure fresh metrics
y_pred_lstm = best_model.predict(X_test, verbose=0)
y_pred_classes_lstm = (y_pred_lstm[:, 1] > THRESHOLD_OPTIMAL).astype(int)
y_true = y_test

# Calculate Metrics for Xception-LSTM
metrics_lstm = {
    'Model': 'Xception-LSTM (Proposed)',
    'Accuracy': accuracy_score(y_true, y_pred_classes_lstm) * 100,
    'Precision': precision_score(y_true, y_pred_classes_lstm, pos_label=1) * 100,
    'Recall': recall_score(y_true, y_pred_classes_lstm, pos_label=1) * 100,
    'Specificity': recall_score(y_true, y_pred_classes_lstm, pos_label=0) * 100,
    'AUC': roc_auc_score(y_true, y_pred_lstm[:, 1]) * 100
}
print(f"   -> Proposed Result: Accuracy={metrics_lstm['Accuracy']:.2f}%, Recall={metrics_lstm['Recall']:.2f}%")

# ==============================================================================
# STEP 2: TRAIN BASELINE MODELS (SPATIAL ONLY)
# ==============================================================================
print("\n--- 2. Training Comparison Models (Spatial Only) ---")

def prepare_static_data(X_video, y_labels):
    """
    Flattens video sequences into individual static frames.
    Example: (N, 15, 224, 224, 3) -> (N*15, 224, 224, 3)
    """
    N_videos, N_frames = X_video.shape[:2]
    X_flat = X_video.reshape(N_videos * N_frames, *X_video.shape[2:])
    y_flat = np.repeat(y_labels, N_frames)
    return X_flat, y_flat

if 'X_train_flat' not in locals():
    print("   -> Converting video data to static frames...")
    X_train_flat, y_train_flat = prepare_static_data(X_train, y_train)
    X_val_flat, y_val_flat = prepare_static_data(X_val, y_val)
    X_test_flat, y_test_flat = prepare_static_data(X_test, y_test)

    y_train_flat_cat = to_categorical(y_train_flat, 2)
    y_val_flat_cat = to_categorical(y_val_flat, 2)
    y_train_flat_int = y_train_flat.astype(int)
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train_flat_int), y=y_train_flat_int)
    cw_dict = dict(enumerate(class_weights))
else:
    print("   -> Static data already prepared.")

def build_baseline_cnn(base_cnn_class):
    """Builds a simple CNN head for static image classification."""
    input_tensor = Input(shape=(224, 224, 3))
    base_model = base_cnn_class(weights='imagenet', include_top=False, input_tensor=input_tensor)
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(2, activation='softmax')(x)
    return Model(inputs=input_tensor, outputs=output)

baseline_results = [metrics_lstm]
baselines = [
    ('MobileNet', MobileNet),
    ('ResNet50', ResNet50),
    ('Xception (Static)', Xception)
]

for name, base_class in baselines:
    print(f"\n   >> Training {name}...")
    model_bl = build_baseline_cnn(base_class)
    model_bl.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

    # Train static baseline
    model_bl.fit(X_train_flat, y_train_flat_cat,
                 validation_data=(X_val_flat, y_val_flat_cat),
                 epochs=EPOCHS_BASELINE, batch_size=16,
                 class_weight=cw_dict, verbose=0)

    # Evaluate static baseline
    y_prob = model_bl.predict(X_test_flat, verbose=0)
    y_cls = np.argmax(y_prob, axis=1)

    res = {
        'Model': name,
        'Accuracy': accuracy_score(y_test_flat, y_cls) * 100,
        'Precision': precision_score(y_test_flat, y_cls, pos_label=1) * 100,
        'Recall': recall_score(y_test_flat, y_cls, pos_label=1) * 100,
        'Specificity': recall_score(y_test_flat, y_cls, pos_label=0) * 100,
        'AUC': roc_auc_score(y_test_flat, y_prob[:, 1]) * 100
    }
    baseline_results.append(res)
    print(f"      -> Done. Acc: {res['Accuracy']:.2f}%, AUC: {res['AUC']:.2f}")

# ==============================================================================
# STEP 3: COMPILATION AND VISUALIZATION
# ==============================================================================
print("\n--- 3. Generating Comparison Charts ---")

df_final = pd.DataFrame(baseline_results)
df_final = df_final.set_index('Model').sort_values(by='Accuracy', ascending=False)

print("\nRESULT TABLE (Copy for Report):")
print(df_final.round(2))

# Melt data for visualization
df_melted = df_final.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')

# Filter specific metrics for plotting
metrics_to_plot = ['Accuracy', 'Recall', 'Precision', 'Specificity', 'AUC']
df_plot = df_melted[df_melted['Metric'].isin(metrics_to_plot)]

plt.figure(figsize=(14, 7))

# Use 'tab10' palette for high-contrast distinctions
sns.barplot(data=df_plot, x='Metric', y='Score', hue='Model', palette='tab10', edgecolor='black')

plt.title('Performance Comparison: Xception-LSTM (Proposed) vs. Baseline Models', fontsize=14, fontweight='bold')
plt.ylabel('Value (%) / AUC (x100)', fontsize=12)
plt.ylim(0, 135)

# Adjust legend position to avoid overlapping data bars
plt.legend(title='Model', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. DATA PREPARATION  ---
# Reset index to move 'Model' from Index back to a Column
df_reset = df_final.reset_index()

# Rename the index column to 'Model' if necessary (in case the index name was lost)
if 'Model' not in df_reset.columns:
    df_reset = df_reset.rename(columns={'index': 'Model'})

# Convert data to 'long' format (melt)
df_final_melted = df_reset.melt(
    id_vars='Model',
    var_name='Metric',
    value_name='Score')

# Filter for specific metrics to plot
metrics_to_plot = ['Accuracy', 'Recall', 'Precision', 'AUC']
df_plot = df_final_melted[df_final_melted['Metric'].isin(metrics_to_plot)]

# --- 2. PLOT COMPARISON BAR CHART ---
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_plot,
    x='Metric',
    y='Score',
    hue='Model',
    palette='viridis'
)

# Set title and labels
plt.title('Comparison Chart of Metrics Across Different Models', fontsize=14)
plt.ylabel('Value (%) / AUC', fontsize=12)
plt.xlabel('Evaluation Metrics', fontsize=12)
plt.ylim(0, 115) # Expand Y-axis to make room for the legend/annotations

# Add horizontal gridlines for readability
plt.grid(axis='y', linestyle='--', alpha=0.7)

# --- FIX: Move Legend outside of the plot area ---
plt.legend(title='Model', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- Evaluating Final Model ---")

# Load the best model saved by the checkpointfrom tensorflow.keras.models import load_model
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix  # Added here
from sklearn.metrics import roc_curve, auc, roc_auc_score

# NEW: Function to calculate Specificity
def calculate_specificity(y_true_labels, y_pred_classes):
    cm = confusion_matrix(y_true_labels, y_pred_classes)
    # TN (Real -> Real) at [0, 0], FP (Real -> Fake) at [0, 1]
    TN = cm[0, 0]
    FP = cm[0, 1]
    if (TN + FP) == 0:
        return 0.0
    return TN / (TN + FP)

print("\n--- Evaluating Final Model (Restored from best epoch) ---")



# Evaluate on the test set
y_pred = best_model.predict(X_test)

THRESHOLD_OPTIMAL = 0.5
y_pred_classes = (y_pred[:, 1] > THRESHOLD_OPTIMAL).astype(int)
y_true = y_test # Using y_test (integer labels)

# Metrics
accuracy = accuracy_score(y_true, y_pred_classes)
specificity_score = calculate_specificity(y_true, y_pred_classes) # NEW: Calculate Specificity

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")
print(f"Specificity: {specificity_score * 100:.2f}%") # NEW: Print Specificity

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=['REAL', 'FAKE']))


In [ ]:
from sklearn.metrics import recall_score, precision_score, accuracy_score
import numpy as np

# Thresholds to test (Starting from 0.5 and decreasing)
THRESHOLDS = [0.5, 0.45, 0.40, 0.35, 0.30]

# y_pred represents raw probabilities from model.predict(X_test)
# y_test represents the true labels (y_true)

print("--- PERFORMANCE ANALYSIS BY CLASSIFICATION THRESHOLD ---")
print("Threshold | Accuracy | Recall (FAKE) | Precision (FAKE) | F1-Score")
print("----------------------------------------------------------------")

for threshold in THRESHOLDS:
    # 1. Apply new threshold: Probability of FAKE > threshold -> FAKE (1)
    # y_pred[:, 1] is the probability of the FAKE class
    y_pred_thresholded = (y_pred[:, 1] > threshold).astype(int)

    # 2. Recalculate metrics
    acc = accuracy_score(y_test, y_pred_thresholded)

    # Calculate Precision/Recall/F1 for the FAKE class (pos_label=1)
    recall_fake = recall_score(y_test, y_pred_thresholded, pos_label=1, zero_division=0)
    precision_fake = precision_score(y_test, y_pred_thresholded, pos_label=1, zero_division=0)

    # Calculate F1-Score (Harmonic mean of Precision and Recall)
    if (precision_fake + recall_fake) == 0:
        f1_score = 0.0
    else:
        f1_score = 2 * (precision_fake * recall_fake) / (precision_fake + recall_fake)

    print(f"{threshold:.2f}    | {acc:.4f}   | {recall_fake:.4f}      | {precision_fake:.4f}        | {f1_score:.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

### 8. PLOTTING CHARTS ###

# Function to plot training history
def plot_training_history(history_dict):
    plt.figure(figsize=(12, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history_dict['accuracy'], label='Train Accuracy')
    plt.plot(history_dict['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(history_dict['loss'], label='Train Loss')
    plt.plot(history_dict['val_loss'], label='Validation Loss')
    plt.title('Model Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Function to plot the Confusion Matrix
def plot_confusion_matrix(y_true_labels, y_pred_classes):
    cm = confusion_matrix(y_true_labels, y_pred_classes)
    cm_labels = ['Real', 'Fake']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()

# Function to plot the ROC Curve
def plot_roc_curve(y_true_labels, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true_labels, y_pred_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR) / Recall')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()

# --- MODIFIED SECTION: LOAD FROM CSV ---
try:
    # Load the CSV file
    history_df = pd.read_csv('/content/drive/MyDrive/DeepfakeData/v2_bidirectional_lstm_400vid/training_history.csv')

    # Convert DataFrame to dictionary format for the plotting function
    combined_history = history_df.to_dict(orient='list')

    print("\n--- Training History Loaded from CSV ---")
    plot_training_history(combined_history)

except FileNotFoundError:
    print("Error: 'training_history.csv' not found. Please check the file path.")
# ---------------------------------------

print("\n--- Confusion Matrix ---")
plot_confusion_matrix(y_test, y_pred_classes)

# Probabilities for the 'Fake' class
y_pred_proba_fake = y_pred[:, 1]

print("\n--- ROC Curve & AUC Score ---")
plot_roc_curve(y_test, y_pred_proba_fake)

In [ ]:
### 9. PREDICTION VISUALIZATION (APPLYING THRESHOLD 0.35) ###
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import numpy as np
import cv2


# Prediction and Visualization function
def predict_and_visualize_frames(video_path, full_model):
    if full_model is None:
        print("Model is not loaded; cannot perform prediction.")
        return

    print(f"--- Analyzing video: {video_path.split('/')[-1]} ---")

    # 1. Frame Extraction
    frames = extract_frames(
        video_path,
        output_size=OUTPUT_FRAME_SIZE,
        frame_count=FRAME_COUNT
    )

    if frames.size == 0:
        print(f"Could not extract frames or no faces were detected.")
        return

    # 2. Data Preparation and Prediction
    processed_frames = frames / 255.0  # Normalize to [0, 1]
    full_sequence_batch = np.expand_dims(processed_frames, axis=0)
    full_prediction = full_model.predict(full_sequence_batch)

    # --- LOGIC: APPLYING THRESHOLD 0.35 ---
    THRESHOLD_RUNTIME = 0.35
    prob_fake = full_prediction[0][1] # Probability of FAKE (Column 1)

    if prob_fake >= THRESHOLD_RUNTIME:
        label = "FAKE"
        # Confidence is the probability of being FAKE
        confidence = prob_fake
    else:
        label = "REAL"
        # Confidence is the probability of being REAL (1 - prob_fake)
        confidence = full_prediction[0][0]

    # 3. Visualizing Frames
    plt.figure(figsize=(20, 8))
    # Using a 3-row, 5-column grid (3, 5) to match FRAME_COUNT=15
    plt.suptitle(f"Frames extracted from: {video_path.split('/')[-1]}", fontsize=16)

    for i in range(FRAME_COUNT):
        ax = plt.subplot(3, 5, i + 1) # Grid layout
        frame_rgb = cv2.cvtColor(frames[i].astype('uint8'), cv2.COLOR_BGR2RGB)
        plt.imshow(frame_rgb)
        plt.title(f"Frame {i+1}"); plt.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # 4. Print final summary results
    print(f"\n==> FINAL RESULT (Threshold {THRESHOLD_RUNTIME}): {label} (Confidence: {confidence * 100:.2f}%)")

# --- EXECUTE PREDICTIONS ---
real_sample_path = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/original/201.mp4"
fake_sample_path = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/Deepfakes/036_035.mp4"

print("\n--- Predicting Real Video ---")
predict_and_visualize_frames(real_sample_path, best_model)

print("\n--- Predicting Fake Video ---")
predict_and_visualize_frames(fake_sample_path, best_model)

In [ ]:
def visualize_model_comparison(real_video_path, fake_video_path, model):
    # 1. Predict on Real Video
    frames_real = extract_frames(real_video_path, output_size=OUTPUT_FRAME_SIZE, frame_count=FRAME_COUNT)
    pred_real = model.predict(np.expand_dims(frames_real / 255.0, axis=0))[0]
    # Calculate confidence based on the winning class
    conf_real = pred_real[0] if pred_real[0] > pred_real[1] else pred_real[1]
    label_real = "REAL" if pred_real[0] > 0.5 else "FAKE" # Using 0.5 threshold

    # 2. Predict on Fake Video
    frames_fake = extract_frames(fake_video_path, output_size=OUTPUT_FRAME_SIZE, frame_count=FRAME_COUNT)
    pred_fake = model.predict(np.expand_dims(frames_fake / 255.0, axis=0))[0]
    conf_fake = pred_fake[1] if pred_fake[1] > 0.5 else pred_fake[0]
    label_fake = "FAKE" if pred_fake[1] > 0.5 else "REAL"

    # 3. Plot Comparison
    plt.figure(figsize=(12, 6))

    # Display the first frame of the Real Video
    plt.subplot(1, 2, 1)
    if len(frames_real) > 0:
        plt.imshow(cv2.cvtColor(frames_real[0].astype('uint8'), cv2.COLOR_BGR2RGB))
        plt.title(f"Real Video\nPrediction: {label_real}\nConfidence: {conf_real:.2%}",
                  color='green' if label_real=='REAL' else 'red', fontsize=14)
        plt.axis('off')

    # Display the first frame of the Fake Video
    plt.subplot(1, 2, 2)
    if len(frames_fake) > 0:
        plt.imshow(cv2.cvtColor(frames_fake[0].astype('uint8'), cv2.COLOR_BGR2RGB))
        plt.title(f"Fake Video (Deepfake)\nPrediction: {label_fake}\nConfidence: {conf_fake:.2%}",
                  color='green' if label_fake=='FAKE' else 'red', fontsize=14)
        plt.axis('off')

    plt.tight_layout()
    plt.show()


real_vid = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/original/201.mp4"
fake_vid = "/content/drive/MyDrive/DeepfakeData/FaceForensics++_C23/Deepfakes/036_035.mp4"

visualize_model_comparison(real_vid, fake_vid, best_model)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_sampling_diagram(total_frames=150, num_selected=15):
    # Create the figure
    fig, ax = plt.subplots(figsize=(14, 4))

    # 1. Draw the timeline of the entire video (gray line)
    ax.plot([0, total_frames], [1, 1], color='#e0e0e0', linewidth=8,
            label='Full Video Duration', zorder=1)

    # Calculate the step size (identical to your training code logic)
    step = total_frames // num_selected

    # Determine indices of the selected frames
    selected_indices = [i * step for i in range(num_selected)]

    # 2. Draw markers for the selected frames (blue circles)
    ax.scatter(selected_indices, [1]*len(selected_indices), color='#007acc',
               s=150, zorder=5, label='Extracted Frames (Input)')

    # 3. Draw vertical guidelines and annotations
    for i, idx in enumerate(selected_indices):
        # Draw dotted line pointing down
        ax.plot([idx, idx], [1, 0.6], color='#007acc', linestyle=':', linewidth=1.5)

        # Label the Frame sequence (only label some to avoid clutter)
        if i == 0 or i == num_selected-1 or i % 2 == 0:
            ax.text(idx, 0.5, f'F{i+1}\n({idx})', ha='center', va='top',
                    fontsize=9, color='#333333', fontweight='bold')

    # Chart Styling
    ax.set_xlim(-5, total_frames + 5)
    ax.set_ylim(0.4, 1.4)
    ax.set_title(f'Illustration of "Uniform Temporal Sampling" Strategy\n'
                 f'Total: {total_frames} frames | Selected: {num_selected} frames | Step: {step}',
                 fontsize=14, pad=20)
    ax.set_xlabel('Timeline (Frame Index)', fontsize=12)

    # Hide Y-axis and unnecessary spines
    ax.get_yaxis().set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

# Execute the function to draw the diagram
draw_sampling_diagram(total_frames=150, num_selected=15)